In [ ]:
import os
import pandas as pd
import pyodbc as odbc
from datetime import datetime
from re import match

# Pega nome do usuário
usuario = os.getenv("USERNAME") or os.getlogin()

# Conexão com o banco de dados
path = fr'CAMINHO DO ARQUIVO'
os.chdir(path)
df_senhas = pd.read_excel('NOME_DO_ARQUIVO_SENHA.xlsx')
server, database, username, password = df_senhas.iloc[0, 0:4]

conn = odbc.connect(
    f'DRIVER={{BANCO DE DADOS}};'
    f'SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

# Consulta para buscar a data de referência
script = "SELECT MAX(Data_Ref) FROM Base_Historica"
df = pd.read_sql(script, conn)
data_retirar = str(df.iloc[0, 0])

# Consulta principal
script = """
SELECT
    c.Data_Contemplacao,
    a.Sit_Contemplacao,
    a.Data_Ref,
    a.Data_Aloc,
    a.ID_Cota,
    a.Macrosituacao,
    b.Sit_Contrato,
    b.Data_Cancel,
    a.Grana_Total
FROM Base_Historica a
LEFT JOIN Alocacoes b ON a.ID_Cota = b.ID_Cota
LEFT JOIN Contemplacao c ON a.ID_Cota = c.ID_Cota
WHERE a.Data_Aloc != ? AND a.Data_Aloc >= '2024-01-01'
"""
df = pd.read_sql(script, conn, params=[data_retirar])

# Consulta de cancelamentos
script = """
SELECT
    c.Data_Contemplacao,
    a.ID_Cota,
    a.Data_Aloc,
    a.Data_Cancel,
    a.Sit_Contrato,
    a.Grana_Total
FROM Alocacoes a
LEFT JOIN Contemplacao c ON a.ID_Cota = c.ID_Cota AND a.Versao = c.Versao
WHERE a.Data_Aloc >= '2024-01-01'
"""
df2 = pd.read_sql(script, conn)

# Consulta de motivos de cancelamento
script = """
SELECT ID_Cota, Data_Cancel, Motivo_Cancelamento
FROM CancelamentoDECotas
"""
df3 = pd.read_sql(script, conn)

# Junta cancelamentos com motivos
df2 = df2.merge(df3, how='left', on=['ID_Cota', 'Data_Cancel'])
df2 = pd.concat([
    df2[df2['Motivo_Cancelamento'].isna()],
    df2[df2['Motivo_Cancelamento'] == 'INADIMPLÊNCIA']
]).drop_duplicates(subset=['ID_Cota'])

# Análise de inadimplência
df_final = df.copy()
df_final['Data_Ref'] = pd.to_datetime(df_final['Data_Ref'])
df_final['Data_Contemplacao'] = pd.to_datetime(df_final['Data_Contemplacao'])
df_final['Ano_Mes_Contemplacao'] = df_final['Data_Contemplacao'].dt.to_period('M')
df_final['Inadimplente'] = df_final['Macrosituacao'] == 'Inadimplente'

# Calcula diferença de meses entre referência e contemplação
df_final['Mes_Diff'] = df_final.apply(
    lambda row: (row['Data_Ref'].year - row['Data_Contemplacao'].year) * 12 +
                (row['Data_Ref'].month - row['Data_Contemplacao'].month),
    axis=1
)

# Matriz de inadimplência
matriz = pd.pivot_table(
    df_final,
    values='Grana_Total',
    index='Ano_Mes_Contemplacao',
    columns='Mes_Diff',
    aggfunc=lambda x: (x[df_final.loc[x.index, 'Inadimplente']].sum() / x.sum()) * 100
).round(1)

# Renomeia apenas as colunas de Mes_Diff
matriz.columns = ['M' + str(int(col)) for col in matriz.columns]

# Reset index e renomeia a coluna da safra
matriz = matriz.reset_index()
matriz = matriz.rename(columns={'Ano_Mes_Contemplacao': 'Safra Contemplacao'})

# Remove safra específica
matriz = matriz[matriz['Safra Contemplacao'] != '2025-03']

# Filtra colunas desejadas
colunas_desejadas = ['Safra Contemplacao'] + [f'M{i}' for i in range(1, 16)]
matriz = matriz[[col for col in colunas_desejadas if col in matriz.columns]]

# Remove último valor de M2 até M15
for col in matriz.columns:
    if match(fr'M([2-9]|1[0-5])$', col):
        idx = matriz[col].last_valid_index()
        if pd.notna(matriz.at[idx, col]):
            matriz.at[idx, col] = None

# Nome do arquivo com data
data_hoje = datetime.today().strftime('%Y-%m-%d')
nome_arquivo = f'Matriz_Inadimplencia_Safra_{data_hoje}.xlsx'

# Caminho de destino
pasta_destino = fr'CAMINHO PARA SALVAR O ARQUIVO'
caminho_completo = os.path.join(pasta_destino, nome_arquivo)

# Salvar Excel
try:
    matriz.to_excel(caminho_completo, index=False)
    print(f"✅ Arquivo salvo com sucesso em: {caminho_completo}")
except Exception as e:
    print(f"❌ Erro ao salvar o arquivo: {e}")


C:\Users\GabrielHenriqueSilva\AppData\Local\Temp\ipykernel_21548\1801367096.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(script, conn)
C:\Users\GabrielHenriqueSilva\AppData\Local\Temp\ipykernel_21548\1801367096.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(script, conn, params=[data_retirar])
C:\Users\GabrielHenriqueSilva\AppData\Local\Temp\ipykernel_21548\1801367096.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(script, conn)
C:\Users\GabrielHenriqueS

✅ Arquivo salvo com sucesso em: C:\Users\GabrielHenriqueSilva\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\BUSINESS INTELLIGENCE\INDICADORES\NOVA ESTRUTURA\INADIMPLÊNCIA\INADIMPLÊNCIA GERAL\Dados\Dados - Safra Contemplação\Matriz_Inadimplencia_Safra_2025-04-12.xlsx


In [44]:
matriz

,Safra Contemplacao,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,M11,M12,M13,M14,M15
0,2024-01,5.6,5.9,7.0,7.7,8.8,7.0,8.8,7.0,11.0,10.2,8.6,11.5,12.3,13.8,NaN
1,2024-02,6.3,6.7,7.3,5.6,5.1,6.3,5.8,5.2,8.9,10.0,3.3,14.6,11.8,NaN,NaN
2,2024-03,4.9,7.4,10.4,6.3,8.1,6.7,8.8,7.0,8.1,8.8,8.3,8.8,NaN,NaN,NaN
3,2024-04,8.5,7.1,9.3,8.5,8.5,9.0,10.6,11.2,12.3,12.5,15.9,NaN,NaN,NaN,NaN
4,2024-05,5.8,9.2,8.0,7.1,6.3,9.2,10.3,10.5,11.7,10.6,NaN,NaN,NaN,NaN,NaN
5,2024-06,6.0,8.2,9.8,7.6,9.4,11.0,11.6,11.9,11.1,NaN,NaN,NaN,NaN,NaN,NaN
6,2024-07,7.1,10.9,8.2,9.6,9.5,14.7,9.7,11.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-08,4.8,6.4,10.1,8.0,9.1,11.1,12.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2024-09,5.4,9.1,12.1,13.2,14.3,12.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2024-10,6.3,7.1,7.8,10.2,9.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
